# 醫療統計基礎框架

> **課程定位**：理論架構篇（1/5）  
> **先備課程**：`Inferential_Statistics/` 系列（假設檢定、卡方檢定、ANOVA、A/B Testing）  
> **學習路徑**：**01 基礎框架** → 02 臨床試驗 → 03 流行病學 → 04 產後憂鬱案例 → 05 醫療品質

## 學習目標
- 理解醫療統計與商業統計的關鍵差異
- 掌握臨床試驗設計類型與對應的統計方法
- 計算並解讀效應量（Effect Size）在醫療情境的意義
- 進行檢定力分析與樣本數估計
- 了解多重比較校正的必要性

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 醫療統計 vs. 商業統計：關鍵差異

| 面向 | 商業統計 | 醫療統計 |
|------|----------|----------|
| 後果 | 營收/轉換率影響 | **人命攸關** |
| 顯著水準 | α = 0.05 常見 | 部分場景要求 α = 0.01 或更嚴格 |
| 監管要求 | 內部決策 | **FDA / TFDA / EMA** 法規審查 |
| 效應量 | 商業價值判斷 | **臨床顯著性** ≠ 統計顯著性 |
| 樣本 | 通常容易取得大樣本 | 受限於病患招募、倫理考量 |
| 分析原則 | Per-protocol 常見 | **意向治療分析（ITT）** 為金標準 |
| 報告規範 | 內部格式 | **CONSORT、STROBE** 等國際標準 |

### 臨床顯著性 vs. 統計顯著性

> **核心觀念**  
> 一個藥物可能讓血壓降低 1 mmHg（p < 0.001，統計顯著），但臨床上 1 mmHg 的改善幾乎沒有意義。反之，一個新療法可能有 15 mmHg 的臨床改善，但因樣本太小而 p > 0.05。
>
> **醫療決策不能只看 p 值，必須同時考慮效應量和臨床意義。**

## 2. 臨床試驗設計類型

### 研究設計分級（證據等級由高到低）

| 等級 | 設計 | 說明 |
|------|------|------|
| I | **隨機對照試驗（RCT）** | 金標準，隨機分配治療/對照組 |
| II | 世代研究（Cohort） | 追蹤暴露組與非暴露組的結局 |
| III | 病例對照研究（Case-Control） | 回溯性比較患病者與健康者的暴露史 |
| IV | 橫斷面研究（Cross-sectional） | 單一時間點的調查 |
| V | 病例報告 / 專家意見 | 最低證據等級 |

### 試驗類型

| 類型 | 假設 | 統計方法 |
|------|------|----------|
| **優越性試驗** | 新藥 > 對照 | 單尾 t 檢定 |
| **非劣性試驗** | 新藥 ≥ 對照 - δ | TOST 單側 |
| **等效性試驗** | |新藥 - 對照| < δ | TOST 雙側 |

### 關鍵指標

**相對風險（Relative Risk, RR）：**
$$RR = \frac{a/(a+b)}{c/(c+d)}$$

**勝算比（Odds Ratio, OR）：**
$$OR = \frac{ad}{bc}$$

**需要治療人數（NNT, Number Needed to Treat）：**
$$NNT = \frac{1}{|p_1 - p_2|} = \frac{1}{ARR}$$

其中 ARR = Absolute Risk Reduction（絕對風險降低）。

In [ ]:
# 模擬一個簡單的臨床試驗 2x2 表
# 情境：新藥 vs. 安慰劑，結果為「發生不良事件」

np.random.seed(42)

# 2x2 列聯表
#              疾病+    疾病-
# 暴露組(藥物)   a        b
# 對照組(安慰劑) c        d

a, b = 15, 85   # 藥物組：15% 發生事件
c, d = 30, 70   # 安慰劑組：30% 發生事件

n_treatment = a + b
n_control = c + d

# 計算風險指標
risk_treatment = a / n_treatment
risk_control = c / n_control

RR = risk_treatment / risk_control
OR = (a * d) / (b * c)
ARR = risk_control - risk_treatment
NNT = 1 / ARR

print("=" * 50)
print("臨床試驗風險指標計算示範")
print("=" * 50)
print(f"\n           事件+   事件-   合計    風險")
print(f"藥物組     {a:>5d}   {b:>5d}   {n_treatment:>5d}   {risk_treatment:.1%}")
print(f"安慰劑組   {c:>5d}   {d:>5d}   {n_control:>5d}   {risk_control:.1%}")

print(f"\n風險指標：")
print(f"  相對風險 RR  = {RR:.3f} → 藥物組風險是對照組的 {RR:.1%}")
print(f"  勝算比 OR    = {OR:.3f}")
print(f"  絕對風險降低 ARR = {ARR:.1%}")
print(f"  需要治療人數 NNT = {NNT:.1f} → 每治療 {NNT:.0f} 人可多避免 1 例事件")

# Fisher's exact test
odds_ratio, p_value = stats.fisher_exact([[a, b], [c, d]])
print(f"\n  Fisher's exact test: OR = {odds_ratio:.3f}, p = {p_value:.4f}")

# OR 的 95% CI (Woolf's method)
log_or = np.log(OR)
se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lower = np.exp(log_or - 1.96 * se_log_or)
ci_upper = np.exp(log_or + 1.96 * se_log_or)
print(f"  OR 95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

## 3. 效應量（Effect Size）在醫療情境的意義

效應量量化「治療效果的大小」，不受樣本量影響：

### Cohen's d（連續結果）

$$d = \frac{\bar{X}_1 - \bar{X}_2}{s_p}$$

其中 $s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}$ 為合併標準差。

| Cohen's d | 效應大小 | 醫療情境範例 |
|-----------|----------|-------------|
| 0.2 | 小 | 輕微的生活品質改善 |
| 0.5 | 中 | 明顯的症狀緩解 |
| 0.8 | 大 | 顯著的疾病控制 |

> **知識補給站**  
> 在醫療研究中，即使 Cohen's d 只有 0.3（小到中等），如果是對致命疾病的治療效果，臨床上可能非常有價值。效應量的「大小」需要結合臨床情境判斷。

In [ ]:
np.random.seed(42)

# 模擬：降血壓藥物試驗
# 治療組：服藥後收縮壓下降
# 對照組：安慰劑後收縮壓下降

n_per_group = 50
treatment_reduction = np.random.normal(loc=12, scale=8, size=n_per_group)  # 平均降 12 mmHg
placebo_reduction = np.random.normal(loc=4, scale=8, size=n_per_group)     # 平均降 4 mmHg

# Cohen's d
mean_diff = treatment_reduction.mean() - placebo_reduction.mean()
sp = np.sqrt(((n_per_group-1)*treatment_reduction.std(ddof=1)**2 + 
              (n_per_group-1)*placebo_reduction.std(ddof=1)**2) / (2*n_per_group - 2))
cohens_d = mean_diff / sp

# t 檢定
t_stat, p_value = stats.ttest_ind(treatment_reduction, placebo_reduction)

# 95% CI of mean difference
se_diff = sp * np.sqrt(1/n_per_group + 1/n_per_group)
ci_lower = mean_diff - 1.96 * se_diff
ci_upper = mean_diff + 1.96 * se_diff

print("=" * 50)
print("降血壓藥物試驗：效應量分析")
print("=" * 50)
print(f"\n治療組 (n={n_per_group}): 平均降壓 {treatment_reduction.mean():.1f} ± {treatment_reduction.std():.1f} mmHg")
print(f"安慰劑 (n={n_per_group}): 平均降壓 {placebo_reduction.mean():.1f} ± {placebo_reduction.std():.1f} mmHg")
print(f"\n平均差異: {mean_diff:.1f} mmHg, 95% CI: ({ci_lower:.1f}, {ci_upper:.1f})")
print(f"Cohen's d: {cohens_d:.3f} ({'小' if abs(cohens_d)<0.5 else '中' if abs(cohens_d)<0.8 else '大'}效應)")
print(f"t 檢定: t = {t_stat:.3f}, p = {p_value:.6f}")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 分布比較
x = np.linspace(-15, 35, 200)
axes[0].plot(x, stats.norm.pdf(x, treatment_reduction.mean(), treatment_reduction.std()), 
             'b-', linewidth=2, label=f'治療組 (μ={treatment_reduction.mean():.1f})')
axes[0].plot(x, stats.norm.pdf(x, placebo_reduction.mean(), placebo_reduction.std()),
             'r-', linewidth=2, label=f'安慰劑 (μ={placebo_reduction.mean():.1f})')
axes[0].fill_between(x, stats.norm.pdf(x, treatment_reduction.mean(), treatment_reduction.std()), alpha=0.2, color='blue')
axes[0].fill_between(x, stats.norm.pdf(x, placebo_reduction.mean(), placebo_reduction.std()), alpha=0.2, color='red')
axes[0].set_title('收縮壓降低量分布', fontsize=13, fontweight='bold')
axes[0].set_xlabel('收縮壓降低 (mmHg)')
axes[0].set_ylabel('機率密度')
axes[0].legend(fontsize=10)

# 效應量視覺化（forest plot 風格）
axes[1].errorbar(mean_diff, 0, xerr=[[mean_diff-ci_lower], [ci_upper-mean_diff]], 
                  fmt='ko', markersize=10, capsize=8, linewidth=2)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title(f'治療效果 (Cohen\'s d = {cohens_d:.2f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('平均差異 (mmHg)')
axes[1].set_yticks([])
axes[1].text(mean_diff, 0.15, f'{mean_diff:.1f} [{ci_lower:.1f}, {ci_upper:.1f}]', 
             ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 4. 檢定力分析與樣本數估計

### 為什麼需要檢定力分析？

| 錯誤類型 | 定義 | 醫療後果 |
|----------|------|----------|
| Type I (α) | 假陽性（無效藥物判定有效） | 無效藥物上市，浪費資源 |
| Type II (β) | 假陰性（有效藥物判定無效） | 有效藥物被放棄，病患失去治療機會 |
| **檢定力 (1-β)** | 正確偵測到真實效果的機率 | 通常要求 **≥ 80%** |

### 四個關鍵參數（知道任意三個可求第四個）

$$\text{效應量 (d)} \longleftrightarrow \text{樣本數 (n)} \longleftrightarrow \text{顯著水準 (α)} \longleftrightarrow \text{檢定力 (1-β)}$$

> **知識補給站**  
> 在醫療研究中，進行**不夠大**的臨床試驗是一個**倫理問題**——讓病患承受試驗風險，卻因樣本不足而無法得出結論，是對病患的不尊重。

In [ ]:
from statsmodels.stats.power import TTestIndPower, NormalIndPower

power_analysis = TTestIndPower()

# 情境：偵測 5 mmHg 血壓差異，σ=10，需要多少樣本？
effect_size = 5 / 10  # Cohen's d = 0.5
alpha = 0.05
power = 0.80

n_required = power_analysis.solve_power(
    effect_size=effect_size, 
    alpha=alpha, 
    power=power, 
    alternative='two-sided'
)

print("=" * 50)
print("樣本數估計：降血壓臨床試驗")
print("=" * 50)
print(f"\n目標：偵測 5 mmHg 的血壓差異")
print(f"假設：σ = 10 mmHg, Cohen's d = {effect_size}")
print(f"顯著水準 α = {alpha}")
print(f"檢定力 1-β = {power}")
print(f"\n所需樣本數（每組）: {int(np.ceil(n_required))} 人")
print(f"總樣本數: {int(np.ceil(n_required)) * 2} 人")

# 檢定力曲線：不同效應量
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：Power vs. Sample Size
sample_sizes = np.arange(10, 200)
for d in [0.2, 0.5, 0.8]:
    powers = [power_analysis.power(effect_size=d, nobs1=n, alpha=0.05) for n in sample_sizes]
    axes[0].plot(sample_sizes, powers, linewidth=2, label=f'd = {d} ({"小" if d==0.2 else "中" if d==0.5 else "大"})')

axes[0].axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% 檢定力')
axes[0].set_xlabel('每組樣本數', fontsize=12)
axes[0].set_ylabel('檢定力 (1-β)', fontsize=12)
axes[0].set_title('檢定力 vs. 樣本數', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右圖：Power vs. Effect Size (fixed n)
effect_sizes = np.linspace(0.1, 1.5, 100)
for n in [30, 50, 100]:
    powers = [power_analysis.power(effect_size=d, nobs1=n, alpha=0.05) for d in effect_sizes]
    axes[1].plot(effect_sizes, powers, linewidth=2, label=f'n = {n}/組')

axes[1].axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel("效應量 (Cohen's d)", fontsize=12)
axes[1].set_ylabel('檢定力 (1-β)', fontsize=12)
axes[1].set_title('檢定力 vs. 效應量', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 多重比較校正

### 為什麼需要校正？

當同時進行 $m$ 個假設檢定（如測試 20 個生物標記物），犯至少一次 Type I 錯誤的機率：

$$P(\text{至少一個假陽性}) = 1 - (1 - \alpha)^m$$

| 檢定數量 m | 不校正的假陽性風險 |
|-----------|-------------------|
| 1 | 5.0% |
| 5 | 22.6% |
| 10 | 40.1% |
| 20 | **64.2%** |
| 100 | **99.4%** |

### 常用校正方法

| 方法 | 控制指標 | 嚴格程度 | 適用場景 |
|------|----------|----------|----------|
| **Bonferroni** | FWER | 最嚴格 | 少量檢定，安全關鍵 |
| **Holm** | FWER | 較嚴格 | Bonferroni 的改良版 |
| **Benjamini-Hochberg** | FDR | 較寬鬆 | 探索性分析，大量檢定 |

In [ ]:
from statsmodels.stats.multitest import multipletests

np.random.seed(42)

# 模擬：20 個生物標記物，其中 3 個真正與疾病相關
n_markers = 20
n_true = 3
n_samples = 50

# 生成 p-values
p_values = []
true_positives = np.random.choice(n_markers, n_true, replace=False)

for i in range(n_markers):
    if i in true_positives:
        # 真正有差異的標記物
        group1 = np.random.normal(0, 1, n_samples)
        group2 = np.random.normal(0.8, 1, n_samples)  # 真實效應
    else:
        # 無差異的標記物
        group1 = np.random.normal(0, 1, n_samples)
        group2 = np.random.normal(0, 1, n_samples)
    
    _, p = stats.ttest_ind(group1, group2)
    p_values.append(p)

p_values = np.array(p_values)

# 各種校正方法
methods = {
    '未校正': (p_values < 0.05),
    'Bonferroni': multipletests(p_values, alpha=0.05, method='bonferroni')[0],
    'Holm': multipletests(p_values, alpha=0.05, method='holm')[0],
    'BH (FDR)': multipletests(p_values, alpha=0.05, method='fdr_bh')[0],
}

# 結果比較
df_results = pd.DataFrame({
    '標記物': [f'Marker_{i+1}' for i in range(n_markers)],
    'p-value': p_values.round(4),
    '真實狀態': ['有效應' if i in true_positives else '無效應' for i in range(n_markers)],
})

for method, significant in methods.items():
    df_results[method] = ['顯著' if s else '' for s in significant]

print("多重比較校正結果")
print("=" * 80)
display(df_results.sort_values('p-value'))

print("\n摘要：")
for method, significant in methods.items():
    tp = sum(significant[i] and i in true_positives for i in range(n_markers))
    fp = sum(significant[i] and i not in true_positives for i in range(n_markers))
    fn = sum(not significant[i] and i in true_positives for i in range(n_markers))
    print(f"  {method:15s}: 顯著={sum(significant)}, 真陽性={tp}, 假陽性={fp}, 假陰性={fn}")

## 6. 本章小結

| 概念 | 重點 |
|------|------|
| 醫療 vs. 商業 | 臨床顯著性 ≠ 統計顯著性，需報告效應量 |
| 研究設計 | RCT 為金標準，證據等級影響結論強度 |
| 風險指標 | RR（世代研究）、OR（病例對照）、NNT（臨床決策） |
| 效應量 | Cohen's d，需結合臨床情境解讀 |
| 檢定力 | ≥ 80%，樣本不足是倫理問題 |
| 多重校正 | Bonferroni（嚴格）→ Holm → BH-FDR（寬鬆） |

---

## 課堂練習

### 基礎題
1. 某臨床試驗：治療組 100 人中 20 人好轉，對照組 100 人中 10 人好轉。計算 RR、OR、NNT。

### 進階題
2. 你正在規劃一個臨床試驗，預期效應量 d = 0.3（小效應），要求 α = 0.05, power = 0.90。使用 `TTestIndPower` 計算所需樣本數。

### 挑戰題
3. 使用模擬方法驗證 Bonferroni 校正的保守性：
   - 模擬 1000 次實驗，每次進行 20 個 t 檢定（全部 H₀ 為真）
   - 計算在「未校正」和「Bonferroni 校正」下的假陽性率
   - 比較理論值與模擬值

---

> **下一堂課**：[02_臨床試驗假設檢定](./02_臨床試驗假設檢定.ipynb) — 將統計工具應用於真實臨床試驗情境